In [1]:
# ════════════════════════════════════════════════════════════════════════════════════════════
# ScoutIA — TEST : ATTRIBUTION DES ÉQUIPES EN RÉSOLUTION RÉDUITE (09/11/2026)
# Même méthode que l'attribution validée (tenues sur 16 images ; par but : Q1 « qui marque » 8 images, Q2 « qui engage »
# 1 image/s, Q3 arbitrage si Q1 et Q2 divergent), appliquée aux 43 VRAIS buts (heures de la vérité terrain) — SANS rejouer
# le pipeline. Chaque match est fait deux fois : résolution NORMALE (référence) puis RÉDUITE, mêmes images, mêmes questions.
# CRITÈRES FIXÉS AVANT (résolution réduite comparée à la normale, sur l'ensemble des 43 buts) :
#   1. aucune attribution FAUSSE de plus (mauvaise équipe affichée) ;
#   2. au plus 1 but attribué (confirmé ou probable, juste) de moins ;
#   3. tenues : la correspondance avec la vraie tenue de chaque équipe est trouvée sur autant de matchs.
# Coût prévu : ≈ 0,45 $ par match en normale + ≈ 0,25 $ en réduite → ≈ 7 $ au total. Reprise : réponses en
# /kaggle/working/attribution_restest_<match>.json (ajouter la sortie en entrée pour ne rien repayer).
# Entrées : codeprojectai, les 10 vidéos, mziparameter, test_resolution_gemini_cellule.py (pour la vérité terrain).
# ════════════════════════════════════════════════════════════════════════════════════════════
RESOLUTION_REDUITE = "medium"     # "medium" ou "low"
MATCHS_A_TESTER = ["spa", "minerois_ster", "franchimont", "raeren", "andrimont", "p1_minerois", "goe", "stembert",
                   "wanze", "juprelle"]
PLAFOND_TOTAL = 12.0              # $ (sécurité)

import os, glob, json, re, collections, subprocess, sys
try:
    import cv2
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "opencv-python-headless"], check=False); import cv2
import numpy as np
from google import genai
from google.genai import types

# ── vérité terrain et vidéos, reprises du notebook de test (même source que tous les tests) ──
_src = next(iter(sorted(glob.glob("/kaggle/input/**/test_resolution_gemini_cellule.py", recursive=True))
                 + glob.glob("/kaggle/working/test_resolution_gemini_cellule.py")), None)
if not _src:
    raise SystemExit("❌ test_resolution_gemini_cellule.py introuvable (vérité terrain)")
_txt = open(_src, encoding="utf-8").read()


def _bloc(nom):
    i = _txt.index(nom + " = {"); j = _txt.index("{", i); n = 0
    for k in range(j, len(_txt)):
        n += {"{": 1, "}": -1}.get(_txt[k], 0)
        if n == 0:
            return _txt[i:k + 1]


_ns = {}
exec(_txt[_txt.index("V_ = "):_txt.index("\n", _txt.index("V_ = "))], _ns)
for _n in ("MATCHS", "VERITE_EQUIPES", "VRAIES_TENUES_TOUTES"):
    exec(_bloc(_n), _ns)
MATCHS, VERITE, VRAIES_TENUES = _ns["MATCHS"], _ns["VERITE_EQUIPES"], _ns["VRAIES_TENUES_TOUTES"]

# ── clé Gemini : param.env du dataset privé « mziparameter » (comme les autres notebooks), sinon secrets Kaggle ──
_pe = next(iter(glob.glob("/kaggle/input/**/mziparameter/**/param.env", recursive=True)
                + glob.glob("/kaggle/input/**/param.env", recursive=True)), None)
if _pe:
    for _l in open(_pe, encoding="utf-8"):
        _l = _l.strip()
        if _l and not _l.startswith("#") and "=" in _l:
            _k, _v = _l.split("=", 1)
            if _k.strip() == "GEMINI_API_KEY" and not os.environ.get("GEMINI_API_KEY"):
                os.environ["GEMINI_API_KEY"] = _v.strip().strip('"').strip("'")
if not os.environ.get("GEMINI_API_KEY"):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["GEMINI_API_KEY"] = UserSecretsClient().get_secret("GEMINI_API_KEY") or ""
    except Exception:
        pass
if not os.environ.get("GEMINI_API_KEY"):
    raise SystemExit("❌ GEMINI_API_KEY introuvable (ajouter « mziparameter » en entrée)")
CLIENT = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
JET = {"normale": [0, 0, 0], "reduite": [0, 0, 0]}          # questions, jetons entrée, jetons sortie+réflexion
cout = lambda r: JET[r][1] / 1e6 * 0.75 + JET[r][2] / 1e6 * 3.75
fmt = lambda s: f"{int(s)//60:02d}:{int(s)%60:02d}"
sec = lambda h: int(h.split(":")[0]) * 60 + float(h.split(":")[1])

QTENUES = """These 16 images come from an amateur football match video (a camera following the play), taken at different moments.
Identify the OUTFIELD players' kit of each of the two teams (ignore goalkeepers and the referee, describe them separately).
For each team give the main colour(s) of the shirt, the shorts and the socks, in simple English colour words
(e.g. "yellow", "red and white", "dark blue", "light blue", "black"). Also say whether the two teams' shirts share a colour.
Answer ONLY with JSON:
{"team_1": {"shirt": "...", "shorts": "...", "socks": "..."}, "team_2": {"shirt": "...", "shorts": "...", "socks": "..."},
 "goalkeepers": "...", "referee": "...", "shirts_share_a_colour": true|false, "certainty": "high|medium|low"}"""
# Questions identiques à l'attribution validée (notebook de test) : exécution des 4 définitions telles quelles
_q = {}
_i = _txt.index("_AENT = "); _j = _txt.index("\n\n\ndef _aeq")
exec(_txt[_i:_j], _q)
AENT, AQ1, AQ2, AQ3 = _q["_AENT"], _q["_AQ1"], _q["_AQ2"], _q["_AQ3"]
AUTRE = {"A": "B", "B": "A"}


def rappel(a0, a1, b0, b1, toujours=False):
    mots = lambda x: set(x.lower().replace(",", " ").split()) - {"and", "dark", "light", "with"}
    com = mots(a0) & mots(b0)
    if com or toujours:
        t = f" Both teams have {' and '.join(sorted(com))} on their shirts." if com else " Shirts alone can be misleading (shadows, light)."
        return t + f" To tell the teams apart, look at the SHORTS: team A has {a1} shorts, team B has {b1} shorts."
    return ""


def demander(rep, fichier, cle, res, cap, instants, consigne, etiquette=True):
    k = f"{res}|{cle}"
    if k in rep:
        return rep[k]
    if cout("normale") + cout("reduite") > PLAFOND_TOTAL:
        raise SystemExit(f"❌ plafond {PLAFOND_TOTAL} $ atteint")
    ims = []
    for i, t in enumerate(instants, 1):
        cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000)
        ok, img = cap.read()
        img = img.copy() if ok else np.zeros((1080, 1920, 3), np.uint8)
        if etiquette:
            cv2.putText(img, f"{i}  {fmt(t)}", (14, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.8, (0, 255, 255), 4)
        ims.append(types.Part.from_bytes(data=cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 85])[1].tobytes(),
                                         mime_type="image/jpeg"))
    cfg = {"temperature": 0.0, "thinking_config": {"thinking_level": "low"}}
    if res == "reduite":
        cfg["media_resolution"] = f"MEDIA_RESOLUTION_{RESOLUTION_REDUITE.upper()}"
    try:
        r = CLIENT.models.generate_content(model="gemini-3.8-flash", contents=[consigne] + ims, config=cfg)
        um = getattr(r, "usage_metadata", None)
        JET[res][0] += 1; JET[res][1] += int(getattr(um, "prompt_token_count", 0) or 0)
        JET[res][2] += int(getattr(um, "candidates_token_count", 0) or 0) + int(getattr(um, "thoughts_token_count", 0) or 0)
        t_ = r.text or ""
        d, _ = json.JSONDecoder().raw_decode(t_[t_.index("{"):])
    except Exception as e:
        print(f"   ⚠️ {k} : {e}"); return None
    rep[k] = d
    json.dump(rep, open(fichier, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    return d


def eq(d, champ):
    if not d or str(d.get("certainty", "")).lower() == "low":
        return None
    v = str(d.get(champ, "unknown")).upper()
    return v if v in AUTRE else None


mots = lambda x: set(str(x).lower().replace(",", " ").replace("/", " ").split()) - {"and", "with", "dark", "light"}
score_t = lambda det, vrai: len(mots(det[0]) & mots(vrai[0])) * 2 + len(mots(det[1]) & mots(vrai[1]))

BILAN = {"normale": collections.Counter(), "reduite": collections.Counter()}
DETAIL = []
for M in MATCHS_A_TESTER:
    video = MATCHS[M]["video"]
    if not os.path.exists(video):
        print(f"⚠️ {M} : vidéo absente ({video}) — match ignoré"); continue
    fichier = f"/kaggle/working/attribution_restest_{M}.json"
    rep = {}
    for p in sorted(glob.glob(f"/kaggle/input/**/attribution_restest_{M}.json", recursive=True)) + [fichier]:
        if os.path.exists(p):
            rep.update(json.load(open(p, encoding="utf-8")))
    cap = cv2.VideoCapture(video)
    ko = float(MATCHS[M].get("coup_envoi_s") or 0)
    deb = max(ko + 60, 15 * 60); fin = deb + 30 * 60
    buts = sorted(VERITE[M], key=sec)
    print(f"\n{'#' * 90}\n# {M} — {len(buts)} vrais buts\n{'#' * 90}")
    for res in ("normale", "reduite"):
        dt = demander(rep, fichier, f"{M}|TENUES", res, cap, [deb + i * (fin - deb) / 15 for i in range(16)], QTENUES, etiquette=False)
        if not dt:
            print(f"   {res} : tenues non détectées"); BILAN[res]["tenues ratées"] += 1; continue
        T1, T2 = dt.get("team_1", {}), dt.get("team_2", {})
        TD = ((str(T1.get("shirt")), str(T1.get("shorts"))), (str(T2.get("shirt")), str(T2.get("shorts"))))
        d_, i_ = score_t(TD[0], VRAIES_TENUES[M][0]) + score_t(TD[1], VRAIES_TENUES[M][1]), \
                 score_t(TD[0], VRAIES_TENUES[M][1]) + score_t(TD[1], VRAIES_TENUES[M][0])
        carte = None if d_ == i_ else ({"A": "A", "B": "B"} if d_ > i_ else {"A": "B", "B": "A"})
        BILAN[res]["tenues trouvées" if carte else "tenues ratées"] += 1
        print(f"   [{res}] tenues : {TD} (vraies {VRAIES_TENUES[M]}) → {'correspondance OK' if carte else 'CORRESPONDANCE IMPOSSIBLE'}")
        (a0, a1), (b0, b1) = TD
        ff = dict(a0=a0, a1=a1, b0=b0, b1=b1, r=rappel(a0, a1, b0, b1))
        for i, h in enumerate(buts):
            g = sec(h); suiv = sec(buts[i + 1]) if i + 1 < len(buts) else None
            lim = (suiv - 5) if suiv else 10 ** 9
            cle = f"{M}|{h}"
            d1 = demander(rep, fichier, cle + "|Q1", res, cap, [g + d for d in range(-8, 7, 2)], AQ1.format(**ff))
            d2 = demander(rep, fichier, cle + "|Q2", res, cap, list(range(int(g) + 15, int(min(g + 120, lim)) + 1)), AQ2.format(**ff))
            s1 = eq(d1, "scoring_team"); s2 = AUTRE.get(eq(d2, "kickoff_team"))
            pts = collections.Counter()
            if s1: pts[s1] += 2
            if s2: pts[s2] += 2
            d3 = None
            if not (s1 and s2 and s1 == s2):
                d3 = demander(rep, fichier, cle + "|Q3", res, cap,
                              [g + 1 + d for d in range(-8, 7, 2)] + list(range(int(g) + 10, int(min(g + 121, lim)), 2)),
                              AQ3.format(**dict(ff, r=rappel(a0, a1, b0, b1, toujours=True))))
                a3, k3 = eq(d3, "scoring_team"), eq(d3, "kickoff_team")
                if a3: pts[a3] += 1
                if k3: pts[AUTRE[k3]] += 1
            tete = "A" if pts["A"] > pts["B"] else "B" if pts["B"] > pts["A"] else None
            sout = 0
            if tete:
                sout = (s1 == tete) + (s2 == tete) + (bool(d3) and (eq(d3, "scoring_team") == tete or AUTRE.get(eq(d3, "kickoff_team")) == tete))
            ecart = abs(pts["A"] - pts["B"])
            if tete and s1 == tete and s2 == tete and ecart >= 4:
                niv = "confirmé"
            elif tete and ecart >= 2 and sout >= 2:
                niv = "probable"
            else:
                niv, tete = "non attribué", None
            vrai = carte.get(tete) if (carte and tete) else None
            verdict = "—" if not tete else ("?" if not carte else ("juste" if vrai == VERITE[M][h] else "FAUX"))
            BILAN[res][(niv, verdict)] += 1
            DETAIL.append((M, h, res, niv, verdict))
            print(f"   [{res}] but {h} ({VERITE[M][h]}) : {niv} → {verdict}")
    cap.release()
    print(f"   coût cumulé : normale {cout('normale'):.2f} $ ({JET['normale'][0]} q.) | réduite {cout('reduite'):.2f} $ ({JET['reduite'][0]} q.)")

# ── BILAN ──
print(f"\n{'═' * 90}\nBILAN — attribution en résolution NORMALE contre {RESOLUTION_REDUITE.upper()}\n{'═' * 90}")
for res in ("normale", "reduite"):
    b = BILAN[res]
    att = b[("confirmé", "juste")] + b[("probable", "juste")]
    faux = sum(v for k, v in b.items() if isinstance(k, tuple) and k[1] == "FAUX")
    na = sum(v for k, v in b.items() if isinstance(k, tuple) and k[0] == "non attribué")
    print(f"   {res:8s} : attribués justes {att} (confirmés {b[('confirmé', 'juste')]}, probables {b[('probable', 'juste')]}) | "
          f"FAUX {faux} | non attribués {na} | tenues trouvées {b['tenues trouvées']}, ratées {b['tenues ratées']} | "
          f"coût {cout(res):.2f} $ ({JET[res][0]} questions payées)")
_par = {}
for m, h, res, niv, v in DETAIL:
    _par.setdefault((m, h), {})[res] = (niv, v)
print("\n   buts dont le résultat change (normale → réduite) :")
for (m, h), d in _par.items():
    if d.get("normale") != d.get("reduite"):
        print(f"     {m} {h} : {d.get('normale')} → {d.get('reduite')}")
bn, br = BILAN["normale"], BILAN["reduite"]
fn = sum(v for k, v in bn.items() if isinstance(k, tuple) and k[1] == "FAUX")
fr = sum(v for k, v in br.items() if isinstance(k, tuple) and k[1] == "FAUX")
an = bn[("confirmé", "juste")] + bn[("probable", "juste")]; ar = br[("confirmé", "juste")] + br[("probable", "juste")]
ok = fr <= fn and ar >= an - 1 and br["tenues trouvées"] >= bn["tenues trouvées"]
print(f"\n   CRITÈRES : FAUX {fr} ≤ {fn} {'✅' if fr <= fn else '❌'} | attribués justes {ar} ≥ {an} − 1 "
      f"{'✅' if ar >= an - 1 else '❌'} | tenues {br['tenues trouvées']} ≥ {bn['tenues trouvées']} "
      f"{'✅' if br['tenues trouvées'] >= bn['tenues trouvées'] else '❌'}")
print(f"   → {'RÉUSSI : attribution en résolution ' + RESOLUTION_REDUITE + ' possible' if ok else 'ÉCHEC : garder la résolution normale pour l attribution'}")
print(f"   économie par match ≈ {(cout('normale') - cout('reduite')) / max(1, len([m for m in MATCHS_A_TESTER])):.2f} $")


##########################################################################################
# spa — 7 vrais buts
##########################################################################################
   [normale] tenues : (('red', 'white'), ('dark blue and light blue', 'black')) (vraies (('blue and black', 'black'), ('red', 'white'))) → correspondance OK
   [normale] but 20:48 (A) : probable → juste
   [normale] but 49:20 (B) : confirmé → juste
   [normale] but 64:10 (A) : confirmé → juste
   [normale] but 75:03 (A) : non attribué → —
   [normale] but 89:05 (B) : probable → juste
   [normale] but 101:41 (A) : confirmé → juste
   [normale] but 112:45 (A) : confirmé → juste
   [reduite] tenues : (('red', 'white'), ('dark blue', 'black')) (vraies (('blue and black', 'black'), ('red', 'white'))) → correspondance OK
   [reduite] but 20:48 (A) : confirmé → juste
   [reduite] but 49:20 (B) : confirmé → juste
   [reduite] but 64:10 (A) : probable → juste
   [reduite] but 75:03 (A) : non at